In [3]:
!wget https://raw.githubusercontent.com/karpathy/ng-video-lecture/refs/heads/master/input.txt

--2026-03-06 17:29:36--  https://raw.githubusercontent.com/karpathy/ng-video-lecture/refs/heads/master/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... failed: Connection timed out.
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... Read error (Connection timed out) in headers.
Retrying.

--2026-03-06 17:30:13--  (try: 2)  https://raw.githubusercontent.com/karpathy/ng-video-lecture/refs/heads/master/input.txt
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M   372KB/s    in 2.9s    

2026-03-06 17:30:17 (372 KB/s

In [1]:
with open('input_cn.txt', 'r') as f:
    text = f.read()

In [2]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1942


In [3]:
print(text[:100])

人工智能的诞生与发展

人工智能（Artificial Intelligence，简称AI）是一门旨在使机器能够模拟人类智能行为的技术科学。它的起源可以追溯到20世纪中叶，1956年的达特茅斯会议被公


In [4]:
chars = sorted(list(set(text)))
print(''.join(chars))
vocab_size = len(chars)
print(vocab_size)


 -01256789ABCEFGINOPRSTYacdefghilmnoprstuvwxy·“”、。《》一上下不与专世业个中临为主义也习了于些人今仍从他代令以们件任伐优会传伦但位低何作使依保信借偏像允元先入全公共关兴其具内再写军冠冬决准凭出分列则创初利别到制前力务动助勇包化医升单卡卷原参双反发取受变只可叶号各合同后向命和响喜器回因团围图在地场型域基堂处备复外多够大夺奇奠好如始学它安完定实家容寒对导将尝层展工巨己已布带幅平年并序应度建开异弃式强归当很律得循微心必志念思性息情意成我或战所手才扎批技拟括持指挑捉捕据掌探接推提握摒撑播支改效数文断斯新方无日旨时明是景智更最月有望朝期未本术机杂李来构果架标核根框梯棋概模次歌正此步段每比求法注活深渐源溯潜点热焦然版特献率环现球理生用由界疗疾病的益监目相真着矩石研破础硬确示社神离秀私种科积称程究突符第等答简算管类精系素索约级纪线练经结络统继续综编网置翰考者而聚背胜能脑自致色茅荐虽行表被要见规视觉解言计认让训议许论证识诊试诞语说诸调谷贡质赖赛走起足跟路践辅辑达过迎运还这进连追送适逐通速逻遇都采释重量锡键长门问队阵阶际降限险陷随隐难需面革音顺预领题风飞首马驶驾验高麦齐（），；
512


In [5]:
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda l: ''.join([itos[i] for i in l])

sample_sentence = "hii there"
print(encode(sample_sentence))
print(decode(encode(sample_sentence)))

[31, 32, 32, 1, 40, 31, 28, 38, 28]
hii there


In [6]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data)


torch.Size([1942]) torch.int64
tensor([ 72, 205, 283,  ..., 131, 473,  50])


In [7]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [8]:
block_size = 8
train_data[:block_size+1]

tensor([ 72, 205, 283, 406, 345, 435, 338,  57, 147])

In [9]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target is {target}")

when input is tensor([72]) the target is 205
when input is tensor([ 72, 205]) the target is 283
when input is tensor([ 72, 205, 283]) the target is 406
when input is tensor([ 72, 205, 283, 406]) the target is 345
when input is tensor([ 72, 205, 283, 406, 345]) the target is 435
when input is tensor([ 72, 205, 283, 406, 345, 435]) the target is 338
when input is tensor([ 72, 205, 283, 406, 345, 435, 338]) the target is 57
when input is tensor([ 72, 205, 283, 406, 345, 435, 338,  57]) the target is 147


In [10]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # print(f"ix.shape:\n{ix.shape}\n{ix}\n{[data[i:i+block_size] for i in ix]}\n{torch.stack([data[i:i+block_size] for i in ix])}\n=====")
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print(f'inputs:\n{xb.shape}\n{xb}')
print(f'targets:\n{yb.shape}\n{yb}')
print('=====')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")


inputs:
torch.Size([4, 8])
tensor([[203, 361, 389, 397, 391, 390, 298, 161],
        [ 38, 300, 298, 510,  53, 381, 122,  84],
        [  5, 213, 510,  11,  33,  28,  44,  18],
        [204, 335, 120,  69,  53, 194, 345, 283]])
targets:
torch.Size([4, 8])
tensor([[361, 389, 397, 391, 390, 298, 161, 146],
        [300, 298, 510,  53, 381, 122,  84, 363],
        [213, 510,  11,  33,  28,  44,  18,  28],
        [335, 120,  69,  53, 194, 345, 283, 406]])
=====
when input is [203] the target: 361
when input is [203, 361] the target: 389
when input is [203, 361, 389] the target: 397
when input is [203, 361, 389, 397] the target: 391
when input is [203, 361, 389, 397, 391] the target: 390
when input is [203, 361, 389, 397, 391, 390] the target: 298
when input is [203, 361, 389, 397, 391, 390, 298] the target: 161
when input is [203, 361, 389, 397, 391, 390, 298, 161] the target: 146
when input is [38] the target: 300
when input is [38, 300] the target: 298
when input is [38, 300, 298] the t

In [11]:
print(xb) # our input(embedding) to the transformer

tensor([[203, 361, 389, 397, 391, 390, 298, 161],
        [ 38, 300, 298, 510,  53, 381, 122,  84],
        [  5, 213, 510,  11,  33,  28,  44,  18],
        [204, 335, 120,  69,  53, 194, 345, 283]])


In [12]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1377)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B, T) tensor of integers
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            # print(f"B = {B}, T = {T}, C = {C}") # 4, 8, 65
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T + 1)
        return idx   


m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(f"logits.shape: {logits.shape}, loss: {loss}") # -ln(1/512), 约等于 6.238

print(decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))



logits.shape: torch.Size([32, 512]), loss: 7.051459789276123

谷齐题s与面提目置综带降背诸线T版率》荐共务队神望合目谷围践隐冬n性够限型76得信t才助围聚命得r业翰者觉新或着说飞版当
赛·性结程证信源核要数矩断参和翰赖段随仍胜还写此络内带杂先再规，通献世言i石文


In [13]:
# create a pytorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [28]:
batch_size = 32
for steps in range(1000):
    xb, yb = get_batch('train')

    logtis, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    print(loss.item())

1.3322231769561768
1.3879362344741821
1.4017289876937866
1.3676166534423828
1.4689337015151978
1.3632855415344238
1.417145013809204
1.473641276359558
1.3932181596755981
1.3564709424972534
1.4025938510894775
1.3349096775054932
1.4065104722976685
1.4610823392868042
1.359487533569336
1.2981597185134888
1.3964993953704834
1.4477348327636719
1.3100417852401733
1.212181806564331
1.4012348651885986
1.4734083414077759
1.3186593055725098
1.3131024837493896
1.3667024374008179
1.3885701894760132
1.3517991304397583
1.3327734470367432
1.3100727796554565
1.3884843587875366
1.376467227935791
1.4781126976013184
1.3003305196762085
1.331669569015503
1.4140719175338745
1.4465328454971313
1.461025357246399
1.3196766376495361
1.2973061800003052
1.3295024633407593
1.3685563802719116
1.4076281785964966
1.3760898113250732
1.4393465518951416
1.3951696157455444
1.3739500045776367
1.3745057582855225
1.3636667728424072
1.41642165184021
1.3566272258758545
1.458898663520813
1.3941686153411865
1.3153184652328491
1.4

In [31]:
print(decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


Tr架构的代的顺序列中的起心，为核心的场景。智能能主义方7年用。此外，Opensfor的技术的分中叶，人工智能将朝着技术的不足也限制（CN）模拟人工智能够理解决循环神经网络（CN）模型对现望、概念，能
